# Provision an AgentCore Web Search Gateway for Claude Cowork

This notebook is the step-by-step equivalent of `provision.py`. It creates an
**Amazon Bedrock AgentCore Gateway** that exposes the managed **Web Search** tool
over MCP, with a least-privilege IAM service role and (optionally) a Cognito user
pool for the inbound OAuth flow used by the Claude **Bedrock 3P** connector.

It reuses the helper functions in `provision.py`, so behaviour stays in sync with
the CLI. Run the cells in order; the last section tears everything down.

> **Disclaimer:** sample for learning/experimentation, not production. Review and
> adapt before real-world use. Provided "as is" (see `LICENSE`).

**Region:** set `REGION` below to a region where the managed Web Search connector
is available (check the AWS docs). Defaults to `us-east-1`.

## 1. Prerequisites

- AWS credentials configured (e.g. `aws configure` / `AWS_PROFILE`) for an account
  with access to Bedrock AgentCore in your chosen region, able to create IAM
  roles, Cognito user pools, and AgentCore gateways.
- `boto3` installed.

In [ ]:
# Needs boto3 >= 1.43.57 (Web Search 'connector' gateway target).
# If boto3 was already imported in this kernel, restart it after upgrading.
%pip install -q --upgrade -r requirements.txt
import boto3; print('boto3', boto3.__version__)

## 2. Configuration

Edit these values as needed. Leave `AUTH_DISCOVERY_URL` / `AUTH_CLIENT_ID` as
`None` to have the notebook create a Cognito user pool for you; set both to bring
your own OIDC IdP instead.

In [ ]:
REGION = "us-east-1"            # region
GATEWAY_NAME = "WebSearchGateway"
TARGET_NAME = "web-search-tool"
ROLE_NAME = "WebSearchGatewayRole"
PREFIX = "agentcore-websearch"  # name prefix for Cognito resources

# Cognito sign-in user for the Cowork login (authorization-code flow).
# Prompted at runtime so nothing is hardcoded or saved in the notebook.
import getpass
USER_EMAIL = input("Sign-in user email (blank to skip user creation): ").strip() or None
# RECOMMENDED: leave the password blank -> Cognito emails an invitation and the
# user sets their own password at first sign-in. getpass does not echo/store it.
USER_PASSWORD = getpass.getpass("Password (blank = email an invitation instead): ") or None

# Bedrock 3P connector loopback callback port (shown in the connector dialog)
CALLBACK_PORT = 62029

# Optional: federated IdP names (e.g. an IAM Identity Center provider) to add to
# the app client's supported identity providers.
IDENTITY_PROVIDERS = [] # iamidc 

# Bring your own IdP instead of Cognito (set BOTH to skip Cognito creation)
AUTH_DISCOVERY_URL = None
AUTH_CLIENT_ID = None

# Optional: also create a client_credentials client for the browserless test below
WITH_M2M_CLIENT = True



## 3. Set up clients

We import the helpers from `provision.py` and build the boto3 clients. `state`
accumulates the IDs of everything we create and is saved for teardown.

In [ ]:
import provision

c = provision.clients(REGION)
account_id = c["sts"].get_caller_identity()["Account"]
state = {"region": REGION, "created_cognito": False}
print("Account:", account_id, "| Region:", REGION)
print("Make sure Web Search is available in this region (see the AWS docs); otherwise creation will fail.")

## 4. Create the least-privilege IAM service role

The gateway assumes this role to call the managed web-search connector. It grants
only `bedrock-agentcore:InvokeWebSearch` (on the web-search tool ARN) and
`InvokeGateway`, and its trust policy is scoped with `aws:SourceAccount` /
`aws:SourceArn`.

In [ ]:
role_arn = provision.create_service_role(c["iam"], account_id, REGION, ROLE_NAME, state)
role_arn

## 5. Inbound auth: create Cognito (or use your own IdP)

The gateway uses `CUSTOM_JWT` inbound auth. The Claude Bedrock 3P connector needs
the OAuth **authorization-code** grant (a real user signs in via the hosted UI) —
`client_credentials` won't work for the connector. The primary `*-cowork` client
is created with the `openid` scope and the loopback callback.

In [ ]:
if AUTH_DISCOVERY_URL and AUTH_CLIENT_ID:
    discovery_url = AUTH_DISCOVERY_URL
    allowed_clients = [AUTH_CLIENT_ID]
    print("Using your own JWT authorizer (no Cognito created)")
else:
    cog = provision.create_cognito(
        c["cognito"], REGION, PREFIX, extra_callbacks=[],
        with_m2m=WITH_M2M_CLIENT,
        user_email=USER_EMAIL,
        user_password=USER_PASSWORD,
        callback_port=CALLBACK_PORT,
        identity_providers=IDENTITY_PROVIDERS,
        state=state,
    )
    discovery_url = cog["discovery_url"]
    allowed_clients = cog["allowed_clients"]
    state["created_cognito"] = True
    state["cognito"] = cog

discovery_url, allowed_clients

## 6. Create the gateway (MCP + CUSTOM_JWT) and wait until READY

In [ ]:
provision.create_gateway(c["acp"], GATEWAY_NAME, role_arn, discovery_url, allowed_clients, state)
print("Gateway MCP URL:", state["gateway_url"])

## 7. Add the managed web-search connector target

In [ ]:
provision.create_web_search_target(c["acp"], state["gateway_id"], TARGET_NAME, state)
provision.save_state(state)
provision._print_summary(state)

## 8. (Optional) Verify the tool without a browser (M2M)

Requires `WITH_M2M_CLIENT = True`. This uses the `client_credentials` client to
list the gateway's tools — a quick sanity check of the gateway itself. This is
**not** how Cowork connects (Cowork uses the authorization-code flow).

In [ ]:
import json, urllib.request, urllib.parse

if state.get("created_cognito") and state.get("cognito_m2m_client_id"):
    cog = state["cognito"]
    data = urllib.parse.urlencode({
        "grant_type": "client_credentials",
        "client_id": state["cognito_m2m_client_id"],
        "client_secret": state["cognito_m2m_client_secret"],
        "scope": cog["scope"],
    }).encode()
    req = urllib.request.Request(cog["token_endpoint"], data=data,
                                 headers={"Content-Type": "application/x-www-form-urlencoded"})
    token = json.load(urllib.request.urlopen(req))["access_token"]

    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/list"}).encode()
    req = urllib.request.Request(state["gateway_url"], data=body, headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    })
    print(urllib.request.urlopen(req).read().decode()[:2000])
else:
    print("Skipped: set WITH_M2M_CLIENT = True (and re-run) to use this check.")

## 9. Connect Claude Cowork (Bedrock 3P connector)

In Claude, open **Settings → Connectors → Add custom connector** and use the
values printed above:

| Dialog field | Value |
|---|---|
| **URL** | the Gateway MCP URL (ends in `/mcp`) |
| **OAuth Client ID / Secret** | the `*-cowork` client id / secret |
| **Authorization server** | the Cognito **issuer** (the discovery URL without `/.well-known/openid-configuration`) |

The connector redirects to `http://127.0.0.1:<PORT>/callback`. We pre-registered
port `CALLBACK_PORT`. If the dialog shows a **different** port, register it (Cognito
matches the redirect URI exactly):

In [ ]:
# Only needed if the connector dialog shows a port other than CALLBACK_PORT.
NEW_PORT = CALLBACK_PORT  # <-- change to the port Claude shows, then run this cell

if state.get("created_cognito") and NEW_PORT != CALLBACK_PORT:
    pool_id = state["cognito_user_pool_id"]
    client_id = state["cognito_user_client_id"]
    cur = c["cognito"].describe_user_pool_client(UserPoolId=pool_id, ClientId=client_id)["UserPoolClient"]
    new_url = f"http://127.0.0.1:{NEW_PORT}/callback"
    callbacks = list(dict.fromkeys(cur.get("CallbackURLs", []) + [new_url]))
    c["cognito"].update_user_pool_client(
        UserPoolId=pool_id, ClientId=client_id, ClientName=cur["ClientName"],
        CallbackURLs=callbacks,
        AllowedOAuthFlows=cur.get("AllowedOAuthFlows", ["code"]),
        AllowedOAuthScopes=cur.get("AllowedOAuthScopes", []),
        AllowedOAuthFlowsUserPoolClient=cur.get("AllowedOAuthFlowsUserPoolClient", True),
        SupportedIdentityProviders=cur.get("SupportedIdentityProviders", ["COGNITO"]),
        ExplicitAuthFlows=cur.get("ExplicitAuthFlows", ["ALLOW_REFRESH_TOKEN_AUTH", "ALLOW_USER_SRP_AUTH"]),
    )
    print("Registered", new_url)
else:
    print("Nothing to do (port unchanged or using your own IdP).")

## 10. Cleanup

Deletes the target, gateway, the Cognito pool (if created) and the IAM role, then
removes the local state file.

> If you associated an AWS WAF web ACL with the gateway, disassociate it first
> (`aws wafv2 disassociate-web-acl --resource-arn <gateway-arn>`), or the gateway
> delete will fail.

In [ ]:
provision.delete_gateway_and_target(c["acp"], state)
if state.get("created_cognito"):
    provision.delete_cognito(c["cognito"], state)
if state.get("role_name"):
    provision.delete_service_role(c["iam"], state["role_name"])

import os
if os.path.exists(provision.STATE_FILE):
    os.remove(provision.STATE_FILE)
print("Teardown complete.")